# Hello Adapter - Granite Switch with HuggingFace

**Duration:** ~5 min

Minimal example of invoking an **embedded LoRA adapter** inside a **Granite Switch** model, using the HuggingFace backend. This notebook uses the **guardian-core** adapter, which evaluates a message against a safety criterion and returns a structured `yes`/`no` score.

**What you'll learn:**
- How to build a single guardian-core call that scores a user message against a safety criterion and prints a parsed `harmful`/`safe` verdict.
- How to load a composed Granite Switch checkpoint with `transformers`.
- How to activate an adapter by passing `adapter_name=...` to `apply_chat_template`.
- The Guardian prompt protocol - how to frame a criterion so the adapter returns a parseable score.

**Adapters used:** `guardian-core` from the [Guardian](https://huggingface.co/ibm-granite/granitelib-guardian-r1.0) library - a general-purpose safety/risk judge that scores any user-supplied criterion (harm, social bias, jailbreaking, groundedness, ...) as `yes`/`no`.

For the recommended inference path (mellea + vLLM), see [`hello_mellea.ipynb`](./hello_mellea.ipynb). This notebook intentionally uses HuggingFace to show the underlying control-token mechanics.

## Prerequisites

**1 * A composed Granite Switch checkpoint** with the `guardian-core` adapter. The default `MODEL_PATH` below points at the pre-composed `ibm-granite/granite-switch-4.1-3b-preview` on HuggingFace (drawn from the [IBM Granite 4.1 collection](https://huggingface.co/collections/ibm-granite/granite-41-language-models)). To compose your own checkpoint instead, see [`./compose_granite_switch.ipynb`](./compose_granite_switch.ipynb) and point `MODEL_PATH` at its output directory.

**2 * Dependencies** (CUDA GPU required):

In [ ]:
%pip install "granite-switch[hf,compose]" jupyter



Full setup details (GPU sizes, HF auth) are in [`../PREREQUISITES.md`](../PREREQUISITES.md).


## 1 * Imports and configuration
Imports are grouped up front so the full dependency set is visible at a glance. `MODEL_PATH` defaults to the pre-composed `ibm-granite/granite-switch-4.1-3b-preview`; override it with a local directory or a different HF repo via the `MODEL_PATH` env var.

In [ ]:
import os
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import granite_switch.hf  # registers the HF backend with AutoModel/AutoConfig

assert torch.cuda.is_available(), "CUDA GPU required"

In [ ]:
# Path or HF repo of a composed Granite Switch checkpoint with guardian-core.
MODEL_PATH = os.environ.get("MODEL_PATH", "ibm-granite/granite-switch-4.1-3b-preview")

print(f"MODEL_PATH: {MODEL_PATH}")

## 2 * Get the model

`MODEL_PATH` already points at a composed checkpoint - either the pre-composed `ibm-granite/granite-switch-4.1-3b-preview` (default) or a local directory you produced via [`./compose_granite_switch.ipynb`](./compose_granite_switch.ipynb). The `from_pretrained` call below will download it on first use.

## 3 * Load the model
Importing `granite_switch.hf` registers the architecture with `transformers.AutoModelForCausalLM`, so the composed checkpoint loads through the standard HuggingFace API.

In [ ]:
print(f"Loading from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForCausalLM.from_pretrained(MODEL_PATH).eval().to("cuda")

# Inspect which adapters are embedded in this checkpoint.
adapter_names = model.config.adapter_names
print(f"Available adapters: {adapter_names}")

ADAPTER_NAME = "guardian-core" if "guardian-core" in adapter_names else adapter_names[0]
print(f"Using adapter: {ADAPTER_NAME}")

## 4 · Guardian prompt protocol
The `guardian-core` adapter is trained to act as a **judge**: given a `<guardian>` block describing a criterion and a scoring schema, it returns a structured JSON response: `{"score": "yes"}` or `{"score": "no"}`.

In [ ]:
JUDGE_SYSTEM = (
    "As a judge agent, your role is to help assess whether the provided text "
    "meets the given judging criteria, utilizing all available information, "
    "including conversations, documents, and tools."
)


def build_guardian_block(criteria: str) -> str:
    """Build a Guardian adapter prompt block with criteria and scoring schema."""
    schema = "If the text meets the criteria, return 'yes'; otherwise, return 'no'."
    return (
        f"<guardian>{JUDGE_SYSTEM}\n\n"
        f"### Criteria: {criteria}\n\n"
        f"### Scoring Schema: {schema}"
    )


print(build_guardian_block("harm")[:300], "...")

## 5 * Invoke the adapter
This is the key moment: `adapter_name=ADAPTER_NAME` tells `apply_chat_template` to insert the adapter's control token into the prompt. At inference time the Granite Switch model reads that control token and routes the relevant LoRA weights into attention.

In [ ]:
test_message = "Hello, how are you today?"
criteria = (
    "Texts deemed universally harmful are those that pose harm across all "
    "contexts and audiences."
)

messages = [
    {"role": "user", "content": test_message},
    {"role": "user", "content": build_guardian_block(criteria)},
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
    adapter_name=ADAPTER_NAME,
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False)

adapter_output = tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
)
print(f"Input text : {test_message!r}")
print(f"Raw output : {adapter_output!r}")

## 6 · Parse the score
The adapter emits JSON: `{"score": "yes"}` or `{"score": "no"}`. Parse the JSON and extract the score, with a fallback to substring matching if the output is malformed.

In [ ]:
import json


def parse_guardian_output(text: str) -> str | None:
    """Parse Guardian adapter JSON output."""
    try:
        result = json.loads(text.strip())
        score = result.get("score", "").lower()
        if score in ("yes", "no"):
            return score
    except json.JSONDecodeError:
        pass

    # Fallback to substring matching if JSON parsing fails
    low = text.lower()
    if "yes" in low:
        return "yes"
    if "no" in low:
        return "no"
    return None


score = parse_guardian_output(adapter_output)
if score is None:
    print(f"WARNING: could not parse score from {adapter_output!r}")
else:
    verdict = "harmful" if score == "yes" else "safe"
    print(f"Guardian verdict: {score!r} -> {verdict}")

## 7 * Next steps

- **Try the Mellea path.** [`hello_mellea.ipynb`](./hello_mellea.ipynb) runs the same adapter through Mellea's wrappers on vLLM - constrained decoding and output parsing come for free.
- **Go deeper on HF mechanics.** [`granite_switch_with_hf.ipynb`](./granite_switch_with_hf.ipynb) walks through composing a checkpoint and invoking adapters turn-by-turn with the HuggingFace backend.
- **Try a real corpus.** [`rag_101.ipynb`](./rag_101.ipynb) builds a vector corpus and runs an answerability check - the smallest end-to-end RAG demo.
- **Compose your own checkpoint.** [`compose_granite_switch.ipynb`](./compose_granite_switch.ipynb) - pick adapters from the IBM libraries and bake them into a single model.
- **Watch ALORA vs LoRA race.** [`alora_vs_lora_race.ipynb`](./alora_vs_lora_race.ipynb) compares the two activation styles head-to-head on the same workload.